# Retinal CVD: Results Analysis

Loads the artifacts written by the pipeline (`outputs/`) and produces the
figures used in the paper:

1. Mediation effect decomposition (direct vs. indirect)
2. Conformal interval coverage / width
3. Segmentation uncertainty maps

Run the pipeline first:
```
python scripts/run_pipeline.py --config configs/default.yaml --synthetic
```

In [ ]:
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt

OUT = 'outputs'
summary = json.load(open(f'{OUT}/summary.json'))
med = pd.read_csv(f'{OUT}/mediation_results.csv')
feat = pd.read_csv(f'{OUT}/features_table.csv')
med.head()

## Figure 1 — Mediation effect decomposition

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
labels = med['exposure'] + ' -> ' + med['mediator']
x = np.arange(len(med))
ax.bar(x - 0.2, med['direct_effect'], width=0.4, label='Direct (NDE)')
ax.bar(x + 0.2, med['indirect_effect'], width=0.4, label='Indirect (NIE)')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Effect on CVD risk'); ax.legend()
ax.set_title('Causal decomposition of retinal features on CVD risk')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_mediation.png', dpi=150)
plt.show()

## Figure 2 — Conformal coverage vs. target

In [ ]:
cr = summary['conformal_regression']
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['target', 'empirical'],
       [cr['target_coverage'], cr['empirical_coverage']],
       color=['#888', '#2a7'])
ax.set_ylim(0, 1.05); ax.set_ylabel('Coverage')
ax.set_title(f"Conformal coverage (width={cr['mean_interval_width']:.1f})")
plt.tight_layout(); plt.savefig(f'{OUT}/fig_coverage.png', dpi=150)
plt.show()

## Figure 3 — Segmentation uncertainty

In [ ]:
unc = np.load(f'{OUT}/uncertainty.npy')
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, i in zip(axes, range(3)):
    im = ax.imshow(unc[i], cmap='magma'); ax.set_title(f'Image {i} uncertainty')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_uncertainty.png', dpi=150)
plt.show()